In [1]:

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

from ast import literal_eval


In [2]:
data = pd.read_excel('D:\\inventory recommendation\\data set\\transaction_dataset_1200.xlsx')
print("Sample Data:")
print(data.head())

data['products_name'] = data['product_names'].apply(literal_eval)
data['category_name'] = data['categories'].apply(literal_eval)


Sample Data:
  transaction_id user_id      user_name transaction_date  \
0          T0001    U040  Krishna Gupta       05-08-2025   
1          T0002    U017    Aarav Patel       07-05-2024   
2          T0003    U040  Krishna Gupta       16-02-2025   
3          T0004    U019     Sneha Nair       14-10-2024   
4          T0005    U014    Arjun Patel       18-07-2025   

                                         product_ids  \
0                             ['FURN037', 'GROC056']   
1                                         ['BKS020']   
2       ['GROC015', 'FURN014', 'FURN055', 'ELEC046']   
3  ['BKS006', 'GROC030', 'ELEC020', 'GROC005', 'B...   
4                               ['BPC050', 'BPC043']   

                                       product_names  \
0  ['Wayfair Folding Table 37', 'Saffola Butter 56']   
1                       ['Scribbles Non-Fiction 20']   
2  ['Daawat Salt 15', 'West Elm Bookshelf 14', 'A...   
3  ['Pilot Gel Pen Pack', 'IndiaGate Rice 30', 'F...   
4  ['Mayb

In [3]:
user_product = input("Enter the product name you want recommendations for: ").strip()

data_filtered = data[data['products_name'].apply(lambda x: user_product in x)]

if data_filtered.empty:
    print(f"⚠️ No transactions found with '{user_product}'. Check spelling.")
else:
    print(f"✅ Found {len(data_filtered)} transactions containing '{user_product}'.")

✅ Found 8 transactions containing 'Saffola Butter 6'.


In [4]:
all_products = sorted(list({p for sublist in data_filtered['products_name'] for p in sublist}))
basket = pd.DataFrame(0, index=data_filtered['transaction_id'], columns=all_products)

for idx, row in data_filtered.iterrows():
    for p in row['products_name']:
        basket.loc[row['transaction_id'], p] = 1

In [5]:
frequent_itemsets = apriori(basket, min_support=0.1, use_colnames=True)


rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)


rules_filtered = rules[rules['antecedents'].apply(lambda x: user_product in list(x))]

if rules_filtered.empty:
    print(f"\n⚠️ No association rules found for '{user_product}'. Try lowering min_support or min_threshold.")
else:
    print(f"\nAssociation Rules for '{user_product}':")
    print(rules_filtered[['antecedents', 'consequents', 'support', 'confidence', 'lift']])



Association Rules for 'Saffola Butter 6':
                                       antecedents  \
5                               (Saffola Butter 6)   
14                              (Saffola Butter 6)   
22                              (Saffola Butter 6)   
28                              (Saffola Butter 6)   
37                              (Saffola Butter 6)   
..                                             ...   
739      (Saffola Butter 6, Steelcase Recliner 26)   
743  (Saffola Butter 6, Sharpener and Eraser Pack)   
744         (Saffola Butter 6, Floral Print Dress)   
745  (Saffola Butter 6, Kama Ayurveda Lip Balm 53)   
750                             (Saffola Butter 6)   

                                           consequents  support  confidence  \
5                                (Aashirvaad Flour 32)    0.125       0.125   
14                                   (Adidas Shoes 24)    0.250       0.250   
22                               (Ashley Dining Table)    0.125       0

c:\Users\nites\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
c:\Users\nites\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlxtend\frequent_patterns\association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


In [7]:
category_of_input = None
for cats, prods in zip(data['category_name'], data['products_name']):
    if user_product in prods:
        category_of_input = cats[prods.index(user_product)]
        break

if category_of_input:
    same_category_products = {p for cats, prods in zip(data['category_name'], data['products_name'])
                              for p, c in zip(prods, cats) if c == category_of_input}
    
    recommendations = rules_filtered[rules_filtered['consequents'].apply(
        lambda x: list(x)[0] in same_category_products
    )]
else:
    recommendations = rules_filtered.copy()


In [9]:
if recommendations.empty:
    print(f"\nNo same-category recommendations found for '{user_product}'.")
else:
    print(f"\n💡 Recommended products (same category as '{category_of_input}'):")
    for idx, row in recommendations.head(5).iterrows():
        item = list(row['consequents'])[0]
        print(f"➡️ {item} — Confidence: {row['confidence']:.2f}, Lift: {row['lift']:.2f}")


💡 Recommended products (same category as 'Groceries'):
➡️ Aashirvaad Flour 32 — Confidence: 0.12, Lift: 1.00
➡️ Chana Dal 1kg — Confidence: 0.12, Lift: 1.00
➡️ Patanjali Sugar 54 — Confidence: 0.12, Lift: 1.00
➡️ Saffola Butter 26 — Confidence: 0.12, Lift: 1.00
➡️ Aashirvaad Flour 32 — Confidence: 0.50, Lift: 4.00


In [14]:
class FrequentBoughtModel:
    def __init__(self, data):
        self.data = data
        
        print("⏳ Building basket matrix...")

        # Filter very small transactions
        data_filtered = data[data['products_name'].apply(lambda x: len(x) > 1)]
        
        # Get all products
        all_products = sorted(list({p for sublist in data_filtered['products_name'] for p in sublist}))
        
        # Create basket matrix
        basket = pd.DataFrame(False, index=data_filtered['transaction_id'], columns=all_products)
        for idx, row in data_filtered.iterrows():
            for p in row['products_name']:
                basket.loc[row['transaction_id'], p] = True
        
        # Remove extremely rare items
        basket = basket.loc[:, (basket.sum(axis=0) >= 3)]
        
        print("⏳ Running Apriori (may take 5–10 seconds)...")
        frequent_itemsets = apriori(basket, min_support=0.001, use_colnames=True)
        
        if frequent_itemsets.empty:
            print("⚠️ No frequent itemsets found even after filtering.")
            self.rules = pd.DataFrame()
            return
        
        print("⏳ Generating rules...")
        rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
        
        self.rules = rules
        print("✅ Frequent model ready!")
        

    def predict(self, product_name, top_n=5):
        if self.rules.empty:
            return []
        
        rules_filtered = self.rules[
            self.rules['antecedents'].apply(lambda x: product_name in list(x))
        ]
        
        if rules_filtered.empty:
            return []
        
        results = []
        for idx, row in rules_filtered.head(top_n).iterrows():
            item = list(row['consequents'])[0]
            results.append({
                "product": item,
                "confidence": float(row['confidence']),
                "lift": float(row['lift'])
            })
        
        return results



In [15]:
frequent_model = FrequentBoughtModel(data)


⏳ Building basket matrix...
⏳ Running Apriori (may take 5–10 seconds)...
⏳ Generating rules...
✅ Frequent model ready!


In [16]:
joblib.dump(frequent_model, "frequent_model.pkl")


['frequent_model.pkl']